# Experimento do TCC — Text-to-SQL com recuperação de demonstrações

Este notebook não contém lógica do experimento. Ele clona o repositório,
instala, escreve a configuração e chama a linha de comando — a lógica toda
vive no GitHub, versionada.

## Roteiro de execução

O notebook tem duas partes e **a ordem importa**. A parte 1 roda sem GPU
porque ela não gera texto: ela existe para provar que o encanamento está
ligado antes de você gastar cota de GPU descobrindo que um caminho do Drive
estava errado.

| Seção | Parte 1 · sem GPU | Parte 2 · com T4 |
|---|---|---|
| 1 · Drive e dados | **rode** | **rode de novo** |
| 2 · Clonar o código | **rode** | **rode de novo** |
| 3 · Instalar | **rode** | **rode de novo** |
| 4 · Config de fumaça | **rode** | pule |
| 5 · Pipeline sem GPU | **rode** (3 células) | pule |
| — trocar para T4 — | | |
| 6 · Confirmar a GPU | pule | **rode** |
| 7 · Instalar modelo + config | pule | **rode** (2 células) |
| 8 · Calibração | pule | **rode** |
| 9 · Investigar | pule | só se precisar |

**Por que repetir 1, 2 e 3 na parte 2:** trocar o tipo de ambiente reinicia a
sessão e apaga o `/content`. O Drive e o que está nele sobrevivem; o
repositório clonado e os pacotes instalados, não.

**Por que pular 4 e 5 na parte 2:** o `work_dir` aponta para o Drive, então o
manifesto, as partições, o índice do Qdrant e o cache de vetores já estão lá
da parte 1. Não precisa refazer — e a seção 7 sobrescreve a configuração.

## 1 · Montar o Drive e descompactar os dados

Rode nas duas partes. A descompactação se resolve sozinha: se os dados já estiverem lá, ela é pulada.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib

base = pathlib.Path('/content/drive/MyDrive/tcc')
if (base / 'data/spider/tables.json').exists():
    print('dados já descompactados, nada a fazer')
else:
    with zipfile.ZipFile(base / 'spider_tcc.zip') as z:
        z.extractall(base)
    print('descompactado')

print(sorted(p.name for p in (base / 'data/spider').iterdir()))

A última linha tem que mostrar `database`, `dev.json`, `tables.json` e `train_spider.json`.

## 2 · Trazer o código

Rode nas duas partes.

In [ ]:
%cd /content
!rm -rf TCC2
!git clone -q https://github.com/VitorEduardoLimaKenor/TCC2.git
%cd TCC2
!git log --oneline -1

O `rm -rf` é intencional: toda sessão começa do código que está no GitHub. Você nunca edita código aqui dentro — edita na sua máquina, dá `push`, e roda esta célula de novo.

## 3 · Instalar

Rode nas duas partes.

In [ ]:
!pip install -q -e .
!python -c "import tcc_rag; print('tcc_rag ok')"

---
## 4 · Configuração do teste de fumaça

**Só na parte 1.**

`backend: dummy` não carrega modelo nenhum. Esta rodada não mede nada — ela responde a outra pergunta, mais urgente: o encanamento está ligado?

In [ ]:
%%writefile configs/colab.yaml
name: colab-fumaca

paths:
  spider_root: /content/drive/MyDrive/tcc/data/spider
  questions_pt: /content/drive/MyDrive/tcc/data/spider_pt/questions_pt.json
  work_dir: /content/drive/MyDrive/tcc/artifacts

data:
  min_questions_per_db: 90
  min_families_per_db: 25
  pool_ratio: 0.60
  val_ratio: 0.15
  test_ratio: 0.25
  near_duplicate_threshold: 0.92
  seed: 20260904

embedding:
  model: intfloat/multilingual-e5-base
  device: cpu
  batch_size: 64
  normalize: true
  query_prefix: "query: "
  passage_prefix: "passage: "

qdrant:
  mode: local
  path: /content/drive/MyDrive/tcc/artifacts/qdrant
  collection: demos
  distance: Cosine

retrieval:
  k: 4
  n_candidates: 25
  lambda_question: 0.5
  fixed_demo_seed: 20260919

generation:
  backend: dummy
  model: none
  dtype: float16
  max_context_tokens: 4096
  max_new_tokens: 256
  seed: 20261010
  batch_size: 8

sandbox:
  timeout_s: 15.0
  max_rows: 5000

## 5 · Rodar o pipeline completo, sem GPU

**Só na parte 1.** São três células.

A indexação baixa o modelo de *embeddings* (~1 GB) e vetoriza 2242 demonstrações na CPU — leve alguns minutos. O resultado fica no Drive, então a parte 2 não precisa repetir.

In [ ]:
!tcc-rag prepare   --config configs/colab.yaml
!tcc-rag partition --config configs/colab.yaml
!tcc-rag leakage   --config configs/colab.yaml

In [ ]:
!tcc-rag index --config configs/colab.yaml

In [ ]:
!tcc-rag run      --config configs/colab.yaml --split val --limit 20
!tcc-rag evaluate --config configs/colab.yaml

### O que confirma que passou

- `bancos elegíveis (>= 90 perguntas, >= 25 famílias): 17`
- `VEREDITO: limpo`
- `indexadas 2242 demonstrações`
- cinco linhas `C0:` a `C4:`, cada uma com `+20 linhas`
- a tabela final com a coluna `n` preenchida

**EX igual a zero é o esperado aqui.** O gerador falso devolve `SELECT 1`.
O que importa é que nada quebrou e que o log foi escrito no Drive.

---
---
# PARE AQUI. Parte 2 · Com GPU

Só continue depois que a parte 1 tiver terminado com `VEREDITO: limpo` e as
cinco condições preenchidas.

### O que fazer agora, na ordem

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar.**
   A sessão reinicia e o `/content` é apagado.
2. Volte ao topo e rode **as seções 1, 2 e 3** de novo.
   A seção 1 vai imprimir "dados já descompactados" — é o esperado.
3. **Pule as seções 4 e 5.** O manifesto, as partições e o índice do Qdrant
   já estão no Drive; a seção 7 sobrescreve a configuração.
4. Siga daqui.

## 6 · Confirmar a GPU

**Só na parte 2.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

Precisa dizer `Tesla T4, 15360 MiB`. Se disser que não há GPU, a cota do Colab gratuito acabou — espere algumas horas ou use o Kaggle.

## 7 · Instalar o suporte a modelo e reescrever a configuração

**Só na parte 2.** São duas células: a instalação e a configuração.

In [ ]:
!pip install -q -e ".[hf]"
!pip install -q bitsandbytes

In [ ]:
%%writefile configs/colab.yaml
name: colab-gpu

paths:
  spider_root: /content/drive/MyDrive/tcc/data/spider
  questions_pt: /content/drive/MyDrive/tcc/data/spider_pt/questions_pt.json
  work_dir: /content/drive/MyDrive/tcc/artifacts

data:
  min_questions_per_db: 90
  min_families_per_db: 25
  pool_ratio: 0.60
  val_ratio: 0.15
  test_ratio: 0.25
  near_duplicate_threshold: 0.92
  seed: 20260904

embedding:
  model: intfloat/multilingual-e5-base
  device: cuda
  batch_size: 128
  normalize: true
  query_prefix: "query: "
  passage_prefix: "passage: "

qdrant:
  mode: local
  path: /content/drive/MyDrive/tcc/artifacts/qdrant
  collection: demos
  distance: Cosine

retrieval:
  k: 4
  n_candidates: 25
  lambda_question: 0.5
  fixed_demo_seed: 20260919

generation:
  backend: hf
  model: Qwen/Qwen2.5-Coder-7B-Instruct
  quantization: bitsandbytes-4bit
  dtype: float16
  max_context_tokens: 4096
  max_new_tokens: 256
  seed: 20261010
  batch_size: 4

sandbox:
  timeout_s: 15.0
  max_rows: 5000

## 8 · A rodada de calibração

**Só na parte 2.**

Este é o passo que separa *"meu número é baixo"* de *"meu código está
quebrado"*, e ele vem antes de qualquer construção em cima.

Roda C0 (sem demonstrações) em inglês. O primeiro comando baixa o modelo
(~5 GB) — leva alguns minutos na primeira vez.

In [ ]:
!tcc-rag run --config configs/colab.yaml --split val --limit 80 \
             --conditions C0 --langs en
!tcc-rag evaluate --config configs/colab.yaml

### Como ler o resultado

| EX de C0 em inglês | Leitura |
|---|---|
| **0,45 – 0,65** | Coerente. Um 7B em 4 bits abaixo de um 14B é esperado. Siga. |
| **0,25 – 0,45** | Baixo demais para ser só o porte. Quase sempre é a extração do SQL da resposta, ou o modelo respondendo com explicação em vez de consulta. |
| **abaixo de 0,20** | Algo quebrado. Suspeitos, nesta ordem: o DDL não chega ao *prompt*, o avaliador compara errado, ou o caminho do SQLite está errado e toda execução falha. |

Carvalho *et al.* reportaram ~0,63 para o Qwen2.5-Coder-14B em inglês.

## 9 · Investigar, se o número não bater

In [ ]:
import json, collections, glob

arq = sorted(glob.glob('/content/drive/MyDrive/tcc/artifacts/runs/*.jsonl'))[-1]
linhas = [json.loads(l) for l in open(arq, encoding='utf-8')]
print(arq, '|', len(linhas), 'linhas')

falhas = collections.Counter(r['exec_error'] for r in linhas if not r['exec_ok'])
print('\nerros de execução mais comuns:')
for erro, n in falhas.most_common(5):
    print(f'  {n:>4}  {erro}')

print('\n--- exemplo de saída bruta do modelo ---')
print(linhas[0]['raw_output'][:600])
print('\n--- SQL extraído ---')
print(linhas[0]['generated_sql'])
print('\n--- SQL de referência ---')
print(linhas[0]['gold_sql'])

---
## Depois disso

Com a calibração coerente, a ordem é: baselines C1, C2 e C3 → método C4 →
varredura de `k`, `n_candidates` e `lambda_question` na validação →
congelar a configuração → teste confirmatório com `--split test`.

O `run` é retomável: se a sessão cair, rode a mesma célula de novo que ele
pula o que já está no log.